In [67]:
# 模型下载地址：https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-funasr-nano-int8-2025-12-30.tar.bz2

# https://huggingface.co/csukuangfj/sherpa-onnx-funasr-nano-int8-2025-12-30

In [3]:
import sherpa_onnx
import soundfile as sf


def create_recognizer(provider="cpu") -> sherpa_onnx.OfflineRecognizer:
    model_dir = Path("../sherpa-onnx-funasr-nano-int8-2025-12-30")
    print(model_dir.absolute())

    tokenizer = model_dir / "Qwen3-0.6B"
    encoder_adaptor = model_dir / "encoder_adaptor.int8.onnx"
    embedding = model_dir / "embedding.int8.onnx"
    llm = model_dir / "llm.int8.onnx"

    return sherpa_onnx.OfflineRecognizer.from_funasr_nano(
        encoder_adaptor=encoder_adaptor.as_posix(),
        llm=llm.as_posix(),
        embedding=embedding.as_posix(),
        tokenizer=tokenizer.as_posix(),
        # num_threads=1,
        # provider=provider,
        debug=True,
        # system_prompt="You are a helpful assistant.",
        # user_prompt="语音转写:",
        # max_new_tokens=512,
        # temperature=1e-6,
        # top_p=0.8,
        # seed=42,
    )


def decode_file(
    recognizer: sherpa_onnx.OfflineRecognizer,
    filename: str,
):
    """Decode a single audio file."""
    audio, sample_rate = sf.read(filename, dtype="float32", always_2d=True)
    audio = audio[:, 0]  # only use the first channel

    stream = recognizer.create_stream()
    stream.accept_waveform(sample_rate, audio)
    recognizer.decode_stream(stream)
    result = stream.result
    return result


In [4]:
recognizer_cpu = create_recognizer()

c:\Users\user0\Documents\CapsWriter-Offline\models\模型测试\..\sherpa-onnx-funasr-nano-int8-2025-12-30


In [5]:
from pathlib import Path

test_wavs_dir = Path("../sherpa-onnx-funasr-nano-int8-2025-12-30/test_wavs")
try:
    for wav_path in test_wavs_dir.glob("rag_*.wav"):
        result = decode_file(recognizer_cpu, wav_path)
        # print(result)
        print(f"{wav_path.name}")
        print(result.text)
        # print(result.timestamps)
        # print(result.tokens)
        print()
except Exception as e:
    print(e)
    print()

rag_biochemistry.wav
results.

rag_chemistry.wav
果然有。

rag_history.wav
，由罗马皇帝钦点的犹地亚王大西律王统治期间。

rag_math.wav
能否被满足的问题。对对对对对对对对对对对。

rag_medical.wav
april肾脏中肾小球囊上的细胞膜孔隙很小

rag_physics.wav
, 于碰撞理论. 月面样本缺少挥发性物质.



In [7]:
import subprocess
import json
from pathlib import Path

models_dir = Path("..")
base_dir = models_dir.parent
funasr_nano_dir = models_dir / "sherpa-onnx-funasr-nano-int8-2025-12-30"
test_wavs_dir = funasr_nano_dir / "test_wavs"

try:
    for wav_path in test_wavs_dir.glob("rag_*.wav"):
        cmd = [
            str(base_dir / ".venv" / "Scripts" / "sherpa-onnx-offline.exe"),
            f"--funasr-nano-encoder-adaptor={funasr_nano_dir / 'encoder_adaptor.int8.onnx'}",
            f"--funasr-nano-llm={funasr_nano_dir / 'llm.int8.onnx'}",
            f"--funasr-nano-tokenizer={funasr_nano_dir / 'Qwen3-0.6B'}",
            f"--funasr-nano-embedding={funasr_nano_dir / 'embedding.int8.onnx'}",
            str(wav_path),
        ]
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )

        # 提取"text"字段
        if result.stderr:
            for line in result.stderr.split("\n"):
                if line.strip().startswith("{") and line.strip().endswith("}"):
                    try:
                        data = json.loads(line.strip())
                        if "text" in data:
                            print(f"{wav_path.name}\n{data['text']}")
                            print()
                    except json.JSONDecodeError:
                        continue

except Exception as e:
    print(f"Error: {e}")

rag_biochemistry.wav
results.

rag_chemistry.wav
某个项目.比如说只在当时被认为是一种含氧酸盐.

rag_history.wav
果然，由罗马皇帝亲点的犹地亚王大西律王统治期间。

rag_math.wav
他们的基本概念。

rag_medical.wav
，内小球囊上的细胞膜孔隙很小。

rag_physics.wav
armondeyeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeondeursomeonde

